# Lun 07/12 · 08:00 — TP v0.11 : découpage en package

Lun 07/12, TP de 08:50 à 09:25, après le CC2. Correction à 09:25, TP fini ou non. La référence v0.11 est publiée à 09:30.

## Objectif

`fleetcheck.py` (300 lignes) devient un package `fleetcheck/` de quatre modules, sans changer son comportement :

- `inventory` lit et valide l'inventaire, et lève `FatalError` au lieu d'appeler `sys.exit` ;
- `checks` contrôle les services, `report` écrit le rapport et l'alerte ;
- `cli` est le seul module qui parle à l'utilisateur et décide du code de sortie ; `python -m fleetcheck` le lance.

Le travail est un déplacement de code : coupez et collez les fonctions de v0.10, puis corrigez les imports. Seule `validate` est réécrite, et elle est fournie à l'étape 1.

## Mode d'emploi du notebook

- Copiez ce notebook à la racine de votre dépôt, à côté de `fleetcheck.py`. Le code s'écrit dans les fichiers de `fleetcheck/`, ouverts dans JupyterLab ; le notebook sert à vérifier chaque étape.
- **Le serveur simulé doit tourner** : `python simserver.py` dans un terminal de JupyterLab, comme en v0.9 et v0.10.
- Exécutez d'abord la cellule d'outils ci-dessous. Chaque vérification affiche `OK`, ou `KO` avec ce qui était attendu.
- Le package s'importe dans le notebook (`from fleetcheck import inventory`). La cellule « Point de départ » active `%autoreload` : une modification enregistrée dans `fleetcheck/` est prise en compte à la cellule suivante, sans redémarrer le noyau.
- `run_module(...)` lance `python -m fleetcheck` comme dans un terminal et renvoie `r.stdout`, `r.stderr` et `r.returncode`. `run(...)`, qui lance `fleetcheck.py`, ne sert plus une fois ce fichier supprimé.
- **Prédire** et **Justifier** marquent les moments clés du TP. Répondez dans la cellule `# Ma réponse :` qui suit, **avant** d'exécuter la suite.
- Rendu en fin de TP : menu *Kernel > Restart Kernel and Run All Cells*, serveur simulé lancé, puis commit et push du dossier `fleetcheck/` et de ce notebook ; `fleetcheck.py` est supprimé par `git rm fleetcheck.py`.

In [ ]:
# Outils du TP : exécuter cette cellule en premier, sans la modifier.
import os
import subprocess
import sys


def check(obtenu, attendu):
    if obtenu == attendu:
        print("OK")
    elif isinstance(attendu, str) and "\n" in attendu:
        print(f"KO, obtenu :\n{obtenu}\nattendu :\n{attendu}")
    else:
        print(f"KO\n  obtenu  : {obtenu!r}\n  attendu : {attendu!r}")


def check_error(fonction, *args, attendu):
    try:
        fonction(*args)
    except Exception as e:
        check(f"{type(e).__name__}: {e}", attendu)
    else:
        print(f"KO, aucune exception levée. Attendu : {attendu}")


def run(*args):
    """Lance fleetcheck.py comme dans un terminal (vu en v0.8 : subprocess)."""
    r = subprocess.run([sys.executable, "fleetcheck.py", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1",
                                              "PYTHONWARNDEFAULTENCODING": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    if "EncodingWarning" in r.stderr:
        print("KO : un open(), read_text() ou subprocess text=True sans encoding= (ligne indiquée dans l'EncodingWarning ci-dessus)")
    return r


def run_module(*args):
    """Lance le package comme dans un terminal : python -m fleetcheck ..."""
    r = subprocess.run([sys.executable, "-m", "fleetcheck", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1",
                                              "PYTHONWARNDEFAULTENCODING": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    if "EncodingWarning" in r.stderr:
        print("KO : un open(), read_text() ou subprocess text=True sans encoding= (ligne indiquée dans l'EncodingWarning ci-dessus)")
    return r

Serveur simulé lancé avec `--base-port 9000` (port `8000` pris) : ajoutez `"--inventory", "servers-sim.csv"` à chaque `run_module()` qui contrôle des services, `--env staging` compris (voir le TP v0.9 pour écrire `servers-sim.csv`).

## Point de départ

Votre `fleetcheck.py` de v0.10, avec `config.json`, `requirements.txt`, `.gitignore` et les inventaires.

En retard ou en panne : copiez dans votre dépôt les fichiers de `reference/` au tag `v0.10` du dépôt public.

Vérifiez avant de commencer :

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import subprocess
import sys
import tempfile
from pathlib import Path

check(Path("fleetcheck.py").exists() or Path("fleetcheck").is_dir(), True)
config = json.loads(Path("config.json").read_text(encoding="utf-8"))
check((config["attempts"], config["reports_dir"]), (3, "reports"))

## Étapes

| Fin | Étape |
|---|---|
| 09:02 | 1. Le package et `inventory.py` |
| 09:15 | 2. `checks.py` et `report.py` |
| 09:25 | 3. `cli.py`, `__main__.py`, suppression de `fleetcheck.py` |

À 09:08, si plus d'un tiers de la salle n'a pas l'étape 1, l'enseignant montre `inventory.py` de la référence ; `checks.py` et `report.py` se copient depuis la référence v0.11 et seule l'étape 3 reste à écrire.

Découpage cible : `fleetcheck/__init__.py` vide, `fleetcheck/__main__.py` qui lance `cli`, et quatre modules. Les flèches sont les imports, absolus (`from fleetcheck import checks, inventory, report` dans `cli.py`) ; seul `__main__` importe `cli`.

```mermaid
flowchart LR
  main_["__main__.py<br>from fleetcheck.cli import main, puis main()"] --> cli["cli.py<br>load_config, run(args), main()"]
  cli --> inv["inventory.py<br>InvalidServer, FatalError (nouvelle), validate, load"]
  cli --> chk["checks.py<br>ping, check_health, fetch_version, latest_release,<br>check_service, global_status, exit_code"]
  cli --> rep["report.py<br>build (ex-build_report), write (ex-write_report),<br>archive_old, as_text, send_alert"]
```

Journal : `log = logging.getLogger(__name__)` dans `checks` et `report` (loggers `fleetcheck.checks`, `fleetcheck.report`, enfants de `fleetcheck`), `logging.getLogger("fleetcheck")` dans `cli`. `inventory` ne journalise plus : il lève des exceptions.

### 1. Le package et `inventory.py` — fin 09:02

- Créez le dossier `fleetcheck/` et un fichier `fleetcheck/__init__.py` vide.
- `fleetcheck/inventory.py` reçoit `InvalidServer`, `validate` et `load`, et une nouvelle exception `class FatalError(Exception): pass`.
- `validate(fields)` reçoit désormais la **liste des champs** d'une ligne, plus un dict : elle contrôle d'abord leur nombre, qui quitte `load`, puis construit le dict. Mêmes messages d'`InvalidServer` qu'en v0.10. Copiez-la telle quelle dans `inventory.py`, avec `COLUMNS` :

```text
COLUMNS = ["name", "env", "ip", "port", "role"]


def validate(fields):
    if len(fields) != len(COLUMNS):
        raise InvalidServer(f"{len(fields)} champs au lieu de {len(COLUMNS)}")
    server = dict(zip(COLUMNS, (f.strip() for f in fields)))
    for key in ("name", "env", "role"):
        if not server[key]:
            raise InvalidServer(f"champ '{key}' vide")
    try:
        ipaddress.ip_address(server["ip"])
    except ValueError:
        raise InvalidServer(f"IP invalide : '{server['ip']}'")
    try:
        server["port"] = int(server["port"])
    except ValueError:
        raise InvalidServer(f"port non entier : '{server['port']}'")
    if not 1 <= server["port"] <= 65535:
        raise InvalidServer(f"port hors de 1-65535 : {server['port']}")
    return server
```

- Dans `load`, chaque ligne non vide passe à `validate(fields)` ; le `dict(zip(...))` et le contrôle du nombre de champs en disparaissent.
- Dans `load`, les erreurs fatales lèvent `FatalError` au lieu de `log.error` + `sys.exit(2)`, avec ces messages : `inventaire illisible : <erreur>`, `en-tête attendu : name,env,ip,port,role`, `aucune ligne valide dans l'inventaire`.
- `load` ne journalise plus les lignes ignorées : elle les renvoie dans `invalid`, et `cli.run` les annonce (étape 3). `inventory` n'importe ni `logging`, ni `sys`, ni `requests`.

**Prédire** — Dans un processus neuf, après `from fleetcheck import inventory`, `"requests" in sys.modules` vaut-il `True` ou `False` ?

In [ ]:
# Ma réponse :

**Justifier** — Pourquoi `load` lève-t-elle `FatalError` au lieu d'appeler `sys.exit(2)` ?

In [ ]:
# Ma réponse :

Vérification : `validate` convertit le port et refuse une ligne de 3 champs, `load` d'un fichier absent lève `FatalError`, et l'import seul d'`inventory` ne charge pas `requests` (vérifié dans un processus neuf, le noyau ayant déjà importé d'autres modules).

In [ ]:
from fleetcheck import inventory

check(inventory.validate(["web-1", "prod", "127.0.0.1", "8001", "web"]),
      {"name": "web-1", "env": "prod", "ip": "127.0.0.1", "port": 8001, "role": "web"})
check_error(inventory.validate, ["web-1", "prod", "127.0.0.1"], attendu="InvalidServer: 3 champs au lieu de 5")
try:
    inventory.load("absent.csv")
except inventory.FatalError as e:
    check(str(e).startswith("inventaire illisible"), True)
else:
    print("KO, aucune FatalError levée")
r = subprocess.run([sys.executable, "-c", "import sys; from fleetcheck import inventory; print('requests' in sys.modules)"],
                   capture_output=True, encoding="utf-8")
check(r.stdout.strip(), "False")

### 2. `checks.py` et `report.py` — fin 09:15

- `fleetcheck/checks.py` : les fonctions de contrôle, avec leurs imports (`platform`, `subprocess`, `time`, `requests`, `logging`) et la constante `SEVERITY`.
- `fleetcheck/report.py` : le rapport, renommé sans répéter le nom du module (`report.build`, `report.write`), l'archivage, la sortie texte et l'alerte.
- Aucun des deux n'importe `cli`.

Vérification :

In [ ]:
from fleetcheck import checks, report

check(checks.exit_code(["ok", "down"], 0), 1)
check(checks.global_status([]), "ok")
with tempfile.TemporaryDirectory() as d:
    check(report.archive_old(d, 7), None)

### 3. `cli.py`, `__main__.py`, suppression de `fleetcheck.py` — fin 09:25

- `fleetcheck/cli.py` : `load_config(path)` lève `FatalError` sur une config illisible. `run(args)` est le corps de l'ancien `main()` et **renvoie** le code de sortie ; `main()` fait l'unique `sys.exit` :

```mermaid
sequenceDiagram
  participant main as cli.main()
  participant run as cli.run(args)
  participant mod as inventory, checks, report
  main->>main: argparse, logging.basicConfig
  main->>run: run(args), dans un try
  run->>run: load_config("config.json")
  run->>mod: inventory.load(args.inventory)
  run->>run: filtre --env
  alt config illisible, inventaire inexploitable ou --env vide
    run-->>main: FatalError (levée par load_config, inventory.load ou le filtre)
    main->>main: log.error("%s", e), puis sys.exit(2)
  else inventaire lu
    run->>run: log.warning("ligne %d ignorée : %s", ...) par ligne de invalid
    run->>mod: contrôles, rapport, alerte
    run-->>main: code de sortie
    main->>main: sys.exit(code)
  end
```
- Le parser se crée par `argparse.ArgumentParser(prog="fleetcheck", description=...)`. Sans `prog`, lancé par `python -m fleetcheck`, Python 3.14 affiche `usage: python -m fleetcheck` : l'outil doit s'annoncer sous son nom, celui de la commande installée en v0.12.
- `fleetcheck/__main__.py` : deux lignes, `from fleetcheck.cli import main` puis `main()`.
- Une fois les vérifications vertes : `git rm fleetcheck.py`.

**Prédire** — Depuis la racine du dépôt, `python fleetcheck/cli.py --env staging` : que se passe-t-il ?

In [ ]:
# Ma réponse :

Vérification : même sortie qu'en v0.10 sur le staging (environ 8 s : 3 essais sur `api-stg-1`), `FatalError` transformée en `ERROR` et code `2`, `fleetcheck.py` supprimé.

```text
ok        web-stg-1    staging
down      api-stg-1    staging  timeout après 3 essai(s)

DOWN : 1 ok, 0 degraded, 1 down, 0 invalide(s)
```

In [ ]:
r = run_module("--env", "staging")
check(r.stdout, """ok        web-stg-1    staging
down      api-stg-1    staging  timeout après 3 essai(s)

DOWN : 1 ok, 0 degraded, 1 down, 0 invalide(s)
""")
check(r.returncode, 1)
r = run_module("--inventory", "absent.csv")
check(((r.stderr.splitlines() or [""])[-1].startswith("ERROR inventaire illisible : "), r.returncode), (True, 2))
r = run_module("--help")
check(r.stdout.startswith("usage: fleetcheck "), True)
check(Path("fleetcheck.py").exists(), False)

## Critère final

`python -m fleetcheck` se comporte comme `fleetcheck.py` de v0.10 : la sortie du staging vérifiée à l'étape 3, et les mêmes codes de sortie : `2` sur l'inventaire cassé (lignes invalides), `0` pour `--help`.

In [ ]:
check(run_module("--inventory", "servers-broken.csv", "--env", "staging").returncode, 2)
check(run_module("--help").returncode, 0)

## Pour aller plus loin

- `python -c "import fleetcheck.cli"` depuis un autre dossier que la racine du dépôt : `ModuleNotFoundError`. Le package n'est trouvable que depuis la racine, tant qu'il n'est pas installé : c'est l'objet de v0.12.
- Ajoutez dans `report.py` un import de `cli` utilisé au niveau du module (`CONFIG = cli.load_config("config.json")`) et lancez `run_module()` : lisez l'`AttributeError` d'import circulaire, puis retirez-le.